In [ ]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_filtered_audio_record
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,37,False)
cnn_model.summary()
#tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch264_valAcc0.9636_valPrec0.7479_valRecall0.6599.h5')#
#cnn_model.load_weights('guitarmidi.weights.h5')


input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/training_subset/training_subset_electric'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_filtered_audio_record).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_data_gen
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

Image height:  148
Before string split: (None, 148, 512), max_x=148.0


Model: "guitar_note_detector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_spectrogram   │ (None, 148, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_mean          │ (None, 148, 256,  │          0 │ input_spectrogra… │
│ (AveragePooling2D)  │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_contrast      │ (None, 148, 256,  │          0 │ input_spectrogra… │
│ (Subtract)          │ 1)                │            │ local_mean[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_2d       │ (None, 148, 256,  │          0 │ local_contrast[0… │
│ (Reshape)           │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_conv… │ (None, 148, 64,   │        136 │ reshape_to_2d[0]… │
│ (Conv2D)            │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_bn    │ (None, 148, 64,   │         32 │ freq_compress_co… │
│ (BatchNormalizatio… │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_act   │ (None, 148, 64,   │          0 │ freq_compress_bn… │
│ (LeakyReLU)         │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_drop  │ (None, 148, 64,   │          0 │ freq_compress_ac… │
│ (SpatialDropout2D)  │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_1d       │ (None, 148, 512)  │          0 │ freq_compress_dr… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ln1      │ (None, 148, 512)  │      1,024 │ reshape_to_1d[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_mha      │ (None, 148, 512)  │    131,776 │ tfm_block1_ln1[0… │
│ (MultiHeadAttentio… │                   │            │ tfm_block1_ln1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_attn_add │ (None, 148, 512)  │          0 │ reshape_to_1d[0]… │
│ (Add)               │                   │            │ tfm_block1_mha[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ln2      │ (None, 148, 512)  │      1,024 │ tfm_block1_attn_… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn1     │ (None, 148, 128)  │     65,664 │ tfm_block1_ln2[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn_drop │ (None, 148, 128)  │          0 │ tfm_block1_ffn1[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn2     │ (None, 148, 512)  │     66,048 │ tfm_block1_ffn_d… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn_add  │ (None, 148, 512)  │          0 │ tfm_block1_attn_

 Total params: 746,806 (2.85 MB)

 Trainable params: 743,462 (2.84 MB)

 Non-trainable params: 3,344 (13.06 KB)

ValueError: A total of 6 objects could not be loaded. Example error message for object <Dense name=chord_block_output_str0, built=True>:

The shape of the target variable and the shape of the target value in `variable.assign(value)` must match. variable.shape=(64, 13), Received: value.shape=(384, 37). Target variable: <Variable path=chord_block_output_str0/kernel, shape=(64, 13), dtype=float32, value=[[-0.02901277  0.02602771 -0.11424613 -0.09338076 -0.13031605  0.02737835
  -0.18233153  0.01099956 -0.09707934  0.2722982  -0.25813788  0.2665266
  -0.18313465]
 [ 0.14095289 -0.09594114 -0.15853703 -0.0210155   0.06319088 -0.08145833
  -0.0226661   0.19773227 -0.23563358 -0.0987847  -0.2526111  -0.22128087
   0.16819352]
 [ 0.13585696 -0.08720361 -0.21308935 -0.2331452  -0.05951166 -0.08548607
  -0.21231565 -0.10146181  0.06831956  0.12796372 -0.17011626 -0.1662479
  -0.20373273]
 [ 0.1445165   0.15849602 -0.01981655 -0.24597917  0.10865119  0.2285622
   0.16524643  0.18731514  0.1642187